In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/processed/combined_dataset_small.csv')
print("Shape:", df.shape)
df.head()

Shape: (1315200, 12)


,zone_id,date,daytype,hour,demand,is_weekend,temp_max,temp_min,precipitation,rain,weather_code,is_raining
0,ZONE_41.7200_-87.6250,2019-01-01,U,0,2.49,1,2.3,-1.5,6.0,3.6,73,1
1,ZONE_41.7200_-87.6250,2019-01-01,U,1,1.25,1,2.3,-1.5,6.0,3.6,73,1
2,ZONE_41.7200_-87.6250,2019-01-01,U,2,0.50,1,2.3,-1.5,6.0,3.6,73,1
3,ZONE_41.7200_-87.6250,2019-01-01,U,3,0.50,1,2.3,-1.5,6.0,3.6,73,1
4,ZONE_41.7200_-87.6250,2019-01-01,U,4,1.25,1,2.3,-1.5,6.0,3.6,73,1


In [2]:
features = ['hour', 'demand', 'is_weekend']

X = df[features]
print("Features shape:", X.shape)
print(X.describe())

Features shape: (1315200, 3)
               hour        demand    is_weekend
count  1.315200e+06  1.315200e+06  1.315200e+06
mean   1.150000e+01  1.708958e+01  3.020073e-01
std    6.922189e+00  2.018218e+01  4.591286e-01
min    0.000000e+00  1.000000e-01  0.000000e+00
25%    5.750000e+00  2.460000e+00  0.000000e+00
50%    1.150000e+01  1.086000e+01  0.000000e+00
75%    1.725000e+01  2.391000e+01  1.000000e+00
max    2.300000e+01  1.823400e+02  1.000000e+00


In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaling complete")
print("Mean:", X_scaled.mean(axis=0))
print("Std:", X_scaled.std(axis=0))

Scaling complete
Mean: [-5.40254513e-21  2.43849277e-16  1.16354615e-16]
Std: [1. 1. 1.]


In [4]:
model = IsolationForest(
    contamination=0.05,
    n_estimators=100,
    random_state=42
)

model.fit(X_scaled)
print("Anomaly model trained successfully")

Anomaly model trained successfully


In [5]:
# Test 1 — Normal rush hour (should be NORMAL = 1)
normal_sample = scaler.transform([[8, 15.0, 0]])
result_normal = model.predict(normal_sample)
print("Rush hour 8AM weekday prediction:", 
      "NORMAL ✅" if result_normal[0] == 1 else "ANOMALY ❌")

# Test 2 — Midnight crowd surge (should be ANOMALY = -1)
anomaly_sample = scaler.transform([[2, 85.0, 0]])
result_anomaly = model.predict(anomaly_sample)
print("Midnight crowd surge prediction:", 
      "ANOMALY ✅" if result_anomaly[0] == -1 else "NORMAL ❌")

# Test 3 — Normal weekend afternoon (should be NORMAL = 1)
weekend_sample = scaler.transform([[14, 12.0, 1]])
result_weekend = model.predict(weekend_sample)
print("Weekend afternoon prediction:", 
      "NORMAL ✅" if result_weekend[0] == 1 else "ANOMALY ❌")

Rush hour 8AM weekday prediction: NORMAL ✅
Midnight crowd surge prediction: ANOMALY ✅
Weekend afternoon prediction: NORMAL ✅


c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [6]:
predictions = model.predict(X_scaled)
total = len(predictions)
anomalies = (predictions == -1).sum()
normal = (predictions == 1).sum()

print("Total rows:", total)
print("Normal rows:", normal)
print("Anomaly rows:", anomalies)
print("Anomaly rate:", round(anomalies/total * 100, 2), "%")

Total rows: 1315200
Normal rows: 1249521
Anomaly rows: 65679
Anomaly rate: 4.99 %


In [7]:
os.makedirs('../models/saved', exist_ok=True)

with open('../models/saved/anomaly_model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'scaler': scaler}, f)

print("Anomaly model saved to models/saved/anomaly_model.pkl")

Anomaly model saved to models/saved/anomaly_model.pkl
